In [26]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn import set_config
from sklearn.model_selection import train_test_split

from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

# Random Surivial Forests

- an ensemble of survival trees
- survival = survival event + time object
- uses the log-rank test for feature & threshold selection

In [27]:
# load datasets
expr = pd.read_csv("../datasets/csv_files/expression_matrix_train.csv")
clinical = pd.read_csv("../datasets/csv_files/clinical_metadata_train.csv")
expr_test = pd.read_csv("../datasets/csv_files/expression_matrix_test_one.csv")
clinical_test = pd.read_csv("../datasets/csv_files/clinical_metadata_test_one.csv")
expr_test.head()

,gene_symbol,GSM540108,GSM540109,GSM540110,GSM540111,GSM540112,GSM540113,GSM540114,GSM540115,GSM540116,...,GSM540364,GSM540365,GSM540366,GSM540367,GSM540368,GSM540369,GSM540370,GSM540371,GSM540372,GSM540373
0,A1BG,6.146570,5.632868,7.407916,5.717083,5.667256,5.821602,5.582540,5.678218,6.570997,...,5.768699,5.915461,5.332402,5.880671,5.785042,6.160731,5.855461,5.794792,5.933755,5.574898
1,A1BG-AS1,5.192952,6.270687,5.594629,4.476920,5.047257,5.420414,4.874277,4.710615,5.320223,...,4.654715,4.821310,4.533131,4.630453,4.559740,4.400831,4.790866,4.519228,4.606000,4.458904
2,A1CF,4.136334,4.220885,4.546149,3.887450,3.876345,3.737222,3.619344,3.747788,3.939471,...,3.730873,3.814982,3.800488,3.891974,3.807973,3.815930,3.925795,3.902201,4.093958,3.588490
3,A2M,7.202705,6.328396,8.172368,8.021876,7.659437,8.497944,8.218462,8.664218,8.092863,...,8.240868,8.781016,8.344433,7.989748,9.159850,8.391154,7.977827,8.225967,8.341430,8.060287
4,A2M-AS1,5.378281,4.429702,6.103873,5.039355,4.568131,5.923578,5.995361,6.468561,4.866476,...,6.963635,5.482779,4.767218,5.862145,5.259731,4.957411,4.985096,5.038558,4.602791,6.741662


In [28]:
# drop non-tumor samples from datasets
cols_to_keep = list(clinical[clinical["is_tumor"] == 1]['sample_name'])
expr_filtered = expr[['gene_symbol'] + cols_to_keep]
clinical_filtered = clinical[clinical["is_tumor"] == 1]

# drop non-tumor samples from datasets
cols_to_keep_test = list(clinical_test[clinical_test["is_tumor"] == 1]['sample_name'])
expr_test_filtered = expr_test[['gene_symbol'] + cols_to_keep_test]
clinical_test_filtered = clinical_test[clinical_test["is_tumor"] == 1]
expr_test_filtered.head()

,gene_symbol,GSM540108,GSM540109,GSM540110,GSM540111,GSM540112,GSM540113,GSM540114,GSM540115,GSM540116,...,GSM540364,GSM540365,GSM540366,GSM540367,GSM540368,GSM540369,GSM540370,GSM540371,GSM540372,GSM540373
0,A1BG,6.146570,5.632868,7.407916,5.717083,5.667256,5.821602,5.582540,5.678218,6.570997,...,5.768699,5.915461,5.332402,5.880671,5.785042,6.160731,5.855461,5.794792,5.933755,5.574898
1,A1BG-AS1,5.192952,6.270687,5.594629,4.476920,5.047257,5.420414,4.874277,4.710615,5.320223,...,4.654715,4.821310,4.533131,4.630453,4.559740,4.400831,4.790866,4.519228,4.606000,4.458904
2,A1CF,4.136334,4.220885,4.546149,3.887450,3.876345,3.737222,3.619344,3.747788,3.939471,...,3.730873,3.814982,3.800488,3.891974,3.807973,3.815930,3.925795,3.902201,4.093958,3.588490
3,A2M,7.202705,6.328396,8.172368,8.021876,7.659437,8.497944,8.218462,8.664218,8.092863,...,8.240868,8.781016,8.344433,7.989748,9.159850,8.391154,7.977827,8.225967,8.341430,8.060287
4,A2M-AS1,5.378281,4.429702,6.103873,5.039355,4.568131,5.923578,5.995361,6.468561,4.866476,...,6.963635,5.482779,4.767218,5.862145,5.259731,4.957411,4.985096,5.038558,4.602791,6.741662


In [29]:
expr_t = expr_filtered.set_index('gene_symbol').T
expr_t.index.name = 'sample_name'
expr_t.reset_index(inplace=True)
expr_t.columns.name = None
expr_t.head()

expr_test_t = expr_test_filtered.set_index('gene_symbol').T
expr_test_t.index.name = 'sample_name'
expr_test_t.reset_index(inplace=True)
expr_test_t.columns.name = None

In [30]:
# add relapse free survival time & event to training data
gene_cols = [col for col in expr_t.columns if col != 'sample_name']

train_data = pd.concat([
        expr_t['sample_name'],
        clinical_filtered[['relapse_free_time', 'relapse_free_event']].astype(int),
        expr_t[gene_cols]
    ], axis=1)

# drop null RFS values & rename columns
train_data.dropna(inplace=True)
train_data[['relapse_free_time', 'relapse_free_event']] = train_data[['relapse_free_time', 'relapse_free_event']].astype(int)

# get gene names
gene_names = list(train_data.columns[3:])

train_data.head()

,sample_name,relapse_free_time,relapse_free_event,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,...,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,mir-223
17,GSM1045225,3026,0,5.006660,4.570530,3.052693,6.212599,4.311186,4.477361,3.720108,...,9.464618,5.080529,3.612147,5.422328,4.565276,5.209277,6.541682,4.941143,6.349697,3.069378
18,GSM1045226,755,1,6.308156,4.849502,3.276010,6.465102,4.034385,3.062556,3.349777,...,7.175808,5.788256,4.008946,5.313596,4.811102,5.625363,5.983060,4.981460,6.539641,2.964171
19,GSM1045227,3014,0,5.043475,4.543174,3.207718,6.398232,3.885093,3.310767,3.272844,...,10.697678,5.242314,3.898463,4.903048,6.729129,5.837191,6.982218,4.759069,6.022257,3.098914
20,GSM1045228,406,1,5.947000,4.793444,3.234156,6.255112,3.975388,3.109538,3.371143,...,8.163624,5.641581,3.613630,5.599734,6.467224,5.726108,5.411201,4.778602,6.210298,2.752077
21,GSM1045229,2225,0,5.389865,4.501806,3.094015,7.015752,3.892831,3.218178,3.337067,...,8.392371,4.979070,3.746636,5.230768,5.193031,5.544478,7.087539,5.130468,6.036339,2.870920


## Training

In [31]:
X_train = train_data[gene_names]
y_train = Surv.from_dataframe('relapse_free_event', 'relapse_free_time', train_data)
random_state = 20

In [32]:
rsf = RandomSurvivalForest(n_estimators=1000, min_samples_split=10, min_samples_leaf=15, n_jobs=-1, random_state=random_state)
rsf.fit(X_train, y_train)

,n_estimators,1000
,max_depth,None
,min_samples_split,10
,min_samples_leaf,15
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,bootstrap,True
,oob_score,False
,n_jobs,-1
,random_state,20


## Testing

In [33]:
# add relapse free survival time & event to testing data
gene_cols = [col for col in expr_test_t.columns if col != 'sample_name']

test_data = pd.concat([
        expr_test_t['sample_name'],
        clinical_test_filtered[['relapse_free_time', 'relapse_free_event']],
        expr_test_t[gene_cols]
    ], axis=1)

# drop null RFS values & rename columns
test_data.dropna(inplace=True)
test_data[['relapse_free_time', 'relapse_free_event']] = test_data[['relapse_free_time', 'relapse_free_event']].astype(int)

# get gene names
gene_names = list(test_data.columns[3:])

test_data.head()

,sample_name,relapse_free_time,relapse_free_event,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,...,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,mir-223
0,GSM540108,731,1,6.146570,5.192952,4.136334,7.202705,5.378281,4.452781,4.456638,...,7.207004,4.956326,4.257066,7.258950,7.369622,6.635957,8.039407,6.266267,6.821724,4.431420
3,GSM540111,1770,1,5.717083,4.476920,3.887450,8.021876,5.039355,3.829404,3.530977,...,8.964749,5.658448,5.376926,6.416896,6.638156,7.852998,9.684551,5.897437,7.499180,4.209039
4,GSM540112,871,1,5.667256,5.047257,3.876345,7.659437,4.568131,3.678200,4.226887,...,9.051191,5.646992,5.464178,6.537697,5.560518,7.861657,8.696353,5.889624,7.680644,3.751116
5,GSM540113,301,1,5.821602,5.420414,3.737222,8.497944,5.923578,3.882873,3.602626,...,7.826577,5.197774,4.890690,6.444662,4.587187,8.174834,10.332553,6.433840,6.908704,4.178111
6,GSM540114,226,1,5.582540,4.874277,3.619344,8.218462,5.995361,3.741898,3.622146,...,9.342628,5.664366,5.156260,6.669061,5.831567,7.816731,9.150694,5.927160,7.267744,4.591735


In [34]:
X_test = test_data[gene_names]
y_test = Surv.from_dataframe('relapse_free_event', 'relapse_free_time', test_data)

In [35]:
c_index = rsf.score(X_test, y_test)
f"{c_index:.5f}"

'0.44096'